In [1]:
!pip install -U \
    langchain \
    langchain-core \
    langchain-community \
    langchain-groq \
    langchain-huggingface \
    sentence-transformers \
    faiss-cpu \
    python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.9
    Uninstalling langchain-core-1

In [2]:
import os

from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

/tmp/ipykernel_3795/1190506230.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


# 1. Load environment variables

In [3]:
GROQ_API_KEY="your_groq_api_key"

# 2. Create sample documents

In [4]:
documents = [
    Document(
        page_content=(
            "LangGraph is a framework for building stateful "
            "and multi-agent AI applications."
        ),
        metadata={"source": "langgraph_notes"},
    ),
       Document(
        page_content=(
            "Inception BD is a Edtech Platform "
            "and provides courses on AI."
        ),
        metadata={"source": "inception_notes"},
    ),
    Document(
        page_content=(
            "RAG stands for Retrieval-Augmented Generation. "
            "It retrieves relevant information before generating an answer."
        ),
        metadata={"source": "rag_notes"},
    ),
    Document(
        page_content=(
            "Groq provides fast inference for supported large "
            "language models through the Groq API."
        ),
        metadata={"source": "groq_notes"},
    ),
    Document(
        page_content=(
            "FAISS is a vector similarity-search library. "
            "It can retrieve documents whose embeddings are close "
            "to the query embedding."
        ),
        metadata={"source": "faiss_notes"},
    ),
]


In [5]:
documents

[Document(metadata={'source': 'langgraph_notes'}, page_content='LangGraph is a framework for building stateful and multi-agent AI applications.'),
 Document(metadata={'source': 'inception_notes'}, page_content='Inception BD is a Edtech Platform and provides courses on AI.'),
 Document(metadata={'source': 'rag_notes'}, page_content='RAG stands for Retrieval-Augmented Generation. It retrieves relevant information before generating an answer.'),
 Document(metadata={'source': 'groq_notes'}, page_content='Groq provides fast inference for supported large language models through the Groq API.'),
 Document(metadata={'source': 'faiss_notes'}, page_content='FAISS is a vector similarity-search library. It can retrieve documents whose embeddings are close to the query embedding.')]

In [6]:
type(documents[0])

langchain_core.documents.base.Document

# 3. Load the embedding model

In [7]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={
        "normalize_embeddings": True,
    },
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

# 4. Create the FAISS vector store and store embeddings

In [8]:
vector_store = FAISS.from_documents(
    documents=documents,
    embedding=embeddings,
)

In [9]:
vector_store.save_local("faiss_index")

In [10]:
vector_store = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

# 5. Create a retriever

In [11]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2},
)

# 6. Initialize the Groq model

In [12]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_retries=2,
    api_key = GROQ_API_KEY,
)

In [13]:
response = llm.invoke("Hi Tell me about you")

In [14]:
response.content

'I\'m an artificial intelligence model known as Llama. Llama stands for "Large Language Model Meta AI."'

# 7. Create the prompt

In [15]:
prompt = ChatPromptTemplate.from_template(
    """
You are a helpful assistant.

Answer the question using only the provided context.

If the answer is not present in the context, say:
"I do not know based on the provided context."

Context:
{context}

Question:
{question}

Answer:
"""
)

# 8. Format retrieved documents

In [16]:
def format_documents(retrieved_documents: list[Document]) -> str:
    return "\n\n".join(
        document.page_content
        for document in retrieved_documents
    )

# 9. Build the RAG chain

In [17]:
rag_chain = (
    {
        "context": retriever | format_documents,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

# 10. Ask questions

In [18]:
def ask_question(question: str) -> str:
    if not question.strip():
        raise ValueError("Question cannot be empty.")

    return rag_chain.invoke(question)

In [19]:
if __name__ == "__main__":
    while True:
        user_question = input(
            "\nAsk a question or type 'exit': "
        ).strip()

        if user_question.lower() == "exit":
            print("Application closed.")
            break

        try:
            answer = ask_question(user_question)

            print("\nAnswer:")
            print(answer)

        except Exception as error:
            print(f"\nError: {error}")


Ask a question or type 'exit': what is faiss

Answer:
FAISS is a vector similarity-search library. It can retrieve documents whose embeddings are close to the query embedding.

Ask a question or type 'exit': what is inception

Answer:
Inception BD is a Edtech Platform and provides courses on AI.

Ask a question or type 'exit': exit
Application closed.
